#Ingest circuits.csv file
- 1.Read the file using spark dataframe reader api
- 2.Add metadata columns ->Source File -> Ingestion timestamp
- 3.Write to Bronze delta table


In [0]:
%run ../00-common/01.environment.config

In [0]:
%run ../00-common/02.bronze_helper

In [0]:
source_file = f"{landing_folder_path}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema_name}.circuits"

#Step 1 - Read the file using spark dataframe reader api

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

circuits_schema = StructType([
StructField('circuitId', StringType()),
StructField('url', StringType()),
StructField('circuitName', StringType()),
StructField('lat', DoubleType()),
StructField('long', DoubleType()),
StructField('locality', StringType()),
StructField('country', StringType())
])

In [0]:
circuits_df = (spark.read
.format('csv')
.option('header','true')
.option('mode', 'FAILFAST')
.schema(circuits_schema)
.load(source_file)
)

In [0]:
display(circuits_df)

circuitId,url,circuitName,lat,long,locality,country
null,https://en.wikipedia.org/wiki/Circuit_Gilles_Villeneuve,circuit gilles villeneuve,45.5,-73.5228,montreal,Canada
null,https://en.wikipedia.org/wiki/Lusail_International_Circuit,losail international circuit,25.49,51.4542,lusail,Qatar
adelaide,https://en.wikipedia.org/wiki/Adelaide_Street_Circuit,adelaide street circuit,-34.9272,138.617,adelaide,Australia
ain-diab,https://en.wikipedia.org/wiki/Ain-Diab_Circuit,ain diab,33.5786,-7.6875,casablanca,Morocco
aintree,https://en.wikipedia.org/wiki/Aintree_Motor_Racing_Circuit,aintree,53.4769,-2.94056,liverpool,UK
albert_park,https://en.wikipedia.org/wiki/Albert_Park_Circuit,albert park grand prix circuit,-37.8497,144.968,melbourne,Australia
americas,https://en.wikipedia.org/wiki/Circuit_of_the_Americas,circuit of the americas,30.1328,-97.6411,austin,USA
anderstorp,https://en.wikipedia.org/wiki/Anderstorp_Raceway,scandinavian raceway,57.2653,13.6042,anderstorp,Sweden
avus,https://en.wikipedia.org/wiki/AVUS,avus,52.4806,13.2514,berlin,Germany
bahrain,https://en.wikipedia.org/wiki/Bahrain_International_Circuit,bahrain international circuit,26.0325,50.5106,sakhir,Bahrain


#Step 2 - Add metadata columns ->Source File -> Ingestion timestamp

In [0]:

circuits_final_df = add_ingestion_metadata(circuits_df)

In [0]:
display(circuits_final_df)

circuitId,url,circuitName,lat,long,locality,country,ingestion_timestamp,source_file
null,https://en.wikipedia.org/wiki/Circuit_Gilles_Villeneuve,circuit gilles villeneuve,45.5,-73.5228,montreal,Canada,2026-06-15T21:45:56.083265Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
null,https://en.wikipedia.org/wiki/Lusail_International_Circuit,losail international circuit,25.49,51.4542,lusail,Qatar,2026-06-15T21:45:56.083265Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
adelaide,https://en.wikipedia.org/wiki/Adelaide_Street_Circuit,adelaide street circuit,-34.9272,138.617,adelaide,Australia,2026-06-15T21:45:56.083265Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
ain-diab,https://en.wikipedia.org/wiki/Ain-Diab_Circuit,ain diab,33.5786,-7.6875,casablanca,Morocco,2026-06-15T21:45:56.083265Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
aintree,https://en.wikipedia.org/wiki/Aintree_Motor_Racing_Circuit,aintree,53.4769,-2.94056,liverpool,UK,2026-06-15T21:45:56.083265Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
albert_park,https://en.wikipedia.org/wiki/Albert_Park_Circuit,albert park grand prix circuit,-37.8497,144.968,melbourne,Australia,2026-06-15T21:45:56.083265Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
americas,https://en.wikipedia.org/wiki/Circuit_of_the_Americas,circuit of the americas,30.1328,-97.6411,austin,USA,2026-06-15T21:45:56.083265Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
anderstorp,https://en.wikipedia.org/wiki/Anderstorp_Raceway,scandinavian raceway,57.2653,13.6042,anderstorp,Sweden,2026-06-15T21:45:56.083265Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
avus,https://en.wikipedia.org/wiki/AVUS,avus,52.4806,13.2514,berlin,Germany,2026-06-15T21:45:56.083265Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
bahrain,https://en.wikipedia.org/wiki/Bahrain_International_Circuit,bahrain international circuit,26.0325,50.5106,sakhir,Bahrain,2026-06-15T21:45:56.083265Z,dbfs:/Volumes/formula1/landing/files/circuits.csv


#Step 3 - Write to Bronze delta table

In [0]:
(
    circuits_final_df
    .write
    .mode('overwrite')
    .format('delta')
    .saveAsTable(table_name)
)

In [0]:
%sql
select * from formula1.bronze.circuits

circuitId,url,circuitName,lat,long,locality,country,ingestion_timestamp,source_file
null,https://en.wikipedia.org/wiki/Circuit_Gilles_Villeneuve,circuit gilles villeneuve,45.5,-73.5228,montreal,Canada,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
null,https://en.wikipedia.org/wiki/Lusail_International_Circuit,losail international circuit,25.49,51.4542,lusail,Qatar,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
adelaide,https://en.wikipedia.org/wiki/Adelaide_Street_Circuit,adelaide street circuit,-34.9272,138.617,adelaide,Australia,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
ain-diab,https://en.wikipedia.org/wiki/Ain-Diab_Circuit,ain diab,33.5786,-7.6875,casablanca,Morocco,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
aintree,https://en.wikipedia.org/wiki/Aintree_Motor_Racing_Circuit,aintree,53.4769,-2.94056,liverpool,UK,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
albert_park,https://en.wikipedia.org/wiki/Albert_Park_Circuit,albert park grand prix circuit,-37.8497,144.968,melbourne,Australia,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
americas,https://en.wikipedia.org/wiki/Circuit_of_the_Americas,circuit of the americas,30.1328,-97.6411,austin,USA,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
anderstorp,https://en.wikipedia.org/wiki/Anderstorp_Raceway,scandinavian raceway,57.2653,13.6042,anderstorp,Sweden,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
avus,https://en.wikipedia.org/wiki/AVUS,avus,52.4806,13.2514,berlin,Germany,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
bahrain,https://en.wikipedia.org/wiki/Bahrain_International_Circuit,bahrain international circuit,26.0325,50.5106,sakhir,Bahrain,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv


In [0]:
display(spark.table('formula1.bronze.circuits'))

circuitId,url,circuitName,lat,long,locality,country,ingestion_timestamp,source_file
null,https://en.wikipedia.org/wiki/Circuit_Gilles_Villeneuve,circuit gilles villeneuve,45.5,-73.5228,montreal,Canada,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
null,https://en.wikipedia.org/wiki/Lusail_International_Circuit,losail international circuit,25.49,51.4542,lusail,Qatar,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
adelaide,https://en.wikipedia.org/wiki/Adelaide_Street_Circuit,adelaide street circuit,-34.9272,138.617,adelaide,Australia,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
ain-diab,https://en.wikipedia.org/wiki/Ain-Diab_Circuit,ain diab,33.5786,-7.6875,casablanca,Morocco,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
aintree,https://en.wikipedia.org/wiki/Aintree_Motor_Racing_Circuit,aintree,53.4769,-2.94056,liverpool,UK,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
albert_park,https://en.wikipedia.org/wiki/Albert_Park_Circuit,albert park grand prix circuit,-37.8497,144.968,melbourne,Australia,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
americas,https://en.wikipedia.org/wiki/Circuit_of_the_Americas,circuit of the americas,30.1328,-97.6411,austin,USA,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
anderstorp,https://en.wikipedia.org/wiki/Anderstorp_Raceway,scandinavian raceway,57.2653,13.6042,anderstorp,Sweden,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
avus,https://en.wikipedia.org/wiki/AVUS,avus,52.4806,13.2514,berlin,Germany,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
bahrain,https://en.wikipedia.org/wiki/Bahrain_International_Circuit,bahrain international circuit,26.0325,50.5106,sakhir,Bahrain,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv


#